# Model Packaging and Deployment

## Introduction

Across Lessons 1–3 you loaded financial data, tested your pipeline, and
built a GARCH model that forecasts volatility. Each piece lives in its own
script or notebook cell — fine for *you*, working interactively. But in a
professional setting, isolated code is not enough.

> ❓ How does someone *else* — a teammate, or another program — train your
> model and request a forecast **without opening your notebook or reading a
> single line of your code**?

The answer is **deployment**: wrap the workflow behind a stable interface
that anything can call. In this lesson you pull all the pieces together —
package the modelling code into a reusable `GarchModel` class, persist
trained models to disk, and expose the whole workflow through a lightweight
REST API built with **FastAPI**.

🎯 **By the end of this notebook you will be able to:**

-   Organise modeling logic into a reusable `GarchModel` class with
    methods for wrangling, fitting, predicting, saving, and loading.
-   Serialize and deserialize trained models using `joblib`.
-   Define request and response schemas with Pydantic data classes.
-   Build a FastAPI application with `/fit` and `/predict` endpoints.
-   Send HTTP requests to your running API and interpret the responses.
-   Debug common issues in module imports, file paths, and API error
    handling.

➡️ Watch the walkthrough, then we turn three lessons of loose code into one
deployable service.

In [1]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1183285385", h="3298dbabb7", width=700, height=450) 

# 1. Conceptual Foundation

## Project context: what we're trying to build

Lessons 1–3 produced three independent capabilities:

-   **Data layer** (`data` module): a `SQLRepository` that reads from and
    writes to a SQLite database.
-   **Configuration** (`config` module): a `settings` object holding the
    database name, API key, and model directory.
-   **Modeling code** (notebook cells): functions that wrangle returns,
    fit a GARCH(1,1) model, clean predictions, and produce a forecast
    dictionary.

This lesson **packages** those capabilities so an API can consume them. The
architecture:

    HTTP client  ──►  FastAPI app (main.py)
                          │
                          ├── /fit   → GarchModel.wrangle → .fit → .dump
                          └── /predict → GarchModel.load → .predict_volatility
                                              │
                                       SQLRepository ◄── SQLite DB

> 🧱 The design is **stateless**: each request creates a fresh
> `GarchModel`, does its work, and returns a JSON response. No
> conversation state lives on the server between calls. That keeps the
> server simple to reason about, trivial to scale (any worker can handle
> any request), and easy to test — exactly the properties production APIs
> are built around.

➡️ The keystone of that architecture is one class that owns the whole
modelling lifecycle.

## Turning notebook code into a Python class

When several functions share the same state — a ticker symbol, a database
connection, a trained model — bundling them into a **class** is the natural
next step. A class folds data (attributes) and behavior (methods) into a
single object.

> 💡 You met this idea in Lesson 2 with `AlphaVantageAPI` and
> `SQLRepository`. Here it returns one level up: `GarchModel` will *own* a
> `SQLRepository` and orchestrate the entire fit → save → load → predict
> cycle through its methods.

A one-cell refresher on the mechanics:

In [2]:
# Quick class refresher with a tiny example
class Greeter:
    """A minimal class to illustrate __init__ and methods."""

    def __init__(self, name):
        self.name = name  # instance attribute

    def greet(self):
        return f"Hello, {self.name}!"

g = Greeter("Volatility")
print(g.greet())
print("Attribute access:", g.name)

Hello, Volatility!
Attribute access: Volatility


For this project the class is `GarchModel`. Its `__init__` stores
configuration, and each method handles one step of the workflow:
`wrangle_data`, `fit`, `predict_volatility`, `dump` (save), and `load`.

> ⚠️ The number-one class bug: forgetting to assign intermediate results to
> `self`. If `fit` computes a trained model but stores it in a *local*
> variable instead of `self.model`, every other method sees nothing —
> `self.model` is still `None`. State that must outlive a method **must**
> live on `self`.

## Serializing models with joblib

Once a model is trained you must **persist** it to disk so the `/predict`
endpoint can reload it later without retraining. The `joblib` library does
this efficiently for scikit-learn–style objects (including `arch` results).

> 📦 Why persist at all? Training a GARCH model on thousands of
> observations takes real time; an API that refit on every prediction
> request would be needlessly slow. Train **once** (`/fit` → `dump`), then
> serve **many** fast predictions (`load` → `predict`). Serialization is
> what decouples the two.

In [3]:
import joblib
import os
from pathlib import Path

# Create a temporary directory for the demo
demo_dir = Path("demo_models")
demo_dir.mkdir(exist_ok=True)

# Save a simple Python object as a stand-in
demo_obj = {"params": [0.01, 0.05, 0.90]}
filepath = demo_dir / "demo_model.pkl"
joblib.dump(demo_obj, filepath)
print("Saved to:", filepath)

# Load it back
loaded = joblib.load(filepath)
print("Loaded:", loaded)

# Clean up
os.remove(filepath)
demo_dir.rmdir()

Saved to: demo_models/demo_model.pkl
Loaded: {'params': [0.01, 0.05, 0.9]}


In the real module you'll build the filename programmatically from a
**timestamp** and the ticker, so every training run writes a uniquely named
file. The `glob` module then finds the most recent file matching a pattern.

In [4]:
from glob import glob

# glob returns a list of paths matching a pattern
# sorted() + [-1] gives the latest file alphabetically
# (works when filenames start with a timestamp)
print(
    "Pattern example:",
    os.path.join("models", "*AMBUJACEM.BSE.pkl"),
)

Pattern example: models/*AMBUJACEM.BSE.pkl


> 📌 The trick that makes "most recent" trivial: because each filename
> *starts* with an ISO 8601 timestamp, plain alphabetical sorting puts the
> newest file last — `sorted(glob(pattern))[-1]`. No date parsing needed.

## Pydantic data classes for request validation

When building an API you must validate that incoming data has the correct
fields and types **before** your code runs. FastAPI integrates tightly with
[Pydantic](https://docs.pydantic.dev/latest/), which lets you declare
expected schemas as Python classes.

In [5]:
from pydantic import BaseModel

class DemoIn(BaseModel):
    ticker: str
    n_days: int

class DemoOut(DemoIn):
    success: bool
    message: str

# Valid input
valid = DemoIn(ticker="AAPL", n_days=5)
print("Valid:", valid)

# Output inherits input fields
out = DemoOut(
    ticker="AAPL", n_days=5,
    success=True, message="ok"
)
print("Output:", out)

Valid: ticker='AAPL' n_days=5
Output: ticker='AAPL' n_days=5 success=True message='ok'


If someone passes the wrong type, Pydantic either coerces it or raises a
clear validation error — your application never sees bad data. Notice that
`DemoOut` **inherits** from `DemoIn` (the `DemoOut(DemoIn)` syntax).
Inheritance means `DemoOut` automatically gets all the fields defined in
`DemoIn` (`ticker` and `n_days`) without rewriting them. You only need to
add the new fields (`success` and `message`). This avoids duplicating
fields and guarantees the response always echoes back the original request
parameters. You will use this same pattern when building `FitOut` and
`PredictOut` later in this lesson.

> 🛡️ Think of a Pydantic model as a **contract at the door**. The schema
> states exactly what a valid request looks like; anything malformed is
> rejected with a precise error *before* it reaches your modelling code. It
> is the request-validation analogue of the `assert`-based tests you wrote
> in Lesson 2.

## FastAPI basics

[FastAPI](https://fastapi.tiangolo.com/) is a modern Python web framework
for building APIs. You define **path functions** decorated with the HTTP
method they handle (`@app.get`, `@app.post`, etc.). FastAPI automatically
generates interactive documentation at `/docs`.

``` python
# Minimal FastAPI example (not executed here — runs on a server)
from fastapi import FastAPI

app = FastAPI()

@app.get("/hello")
def hello():
    return {"message": "Hello from FastAPI!"}
```

You start the server from the terminal with:

``` bash
uvicorn main:app --reload --workers 1 --host localhost --port 8008
```

-   `main:app` — the `app` object inside `main.py`.
-   `--reload` — restart automatically when code changes.
-   `--workers 1` — single worker (enough for development).

Then you interact with the API by sending HTTP requests using the
`requests` library.

## Putting it all together: the deployment flow

1.  A client sends a `POST` request to `/fit` with a JSON body specifying
    the ticker, number of observations, and GARCH parameters.
2.  The server creates a `GarchModel`, wrangles data, fits the model, and
    saves it to disk. It returns a success/failure message.
3.  Later, a client sends a `POST` to `/predict` with the ticker and
    forecast horizon.
4.  The server creates a new `GarchModel`, loads the most recent saved
    model for that ticker, generates predictions, and returns the forecast
    dictionary.

> 🔧 Error handling is non-negotiable: **every** endpoint wraps its work in
> `try`/`except` and returns a structured error response instead of
> crashing. A web server that dies on one bad request takes down every
> other client with it.

## Key points

-   [`joblib.dump` / `joblib.load`](https://joblib.readthedocs.io/en/latest/generated/joblib.dump.html)
    — serialize and deserialize Python objects.
-   [`glob.glob`](https://docs.python.org/3/library/glob.html#glob.glob)
    — find files matching a pattern.
-   [`os.path.join`](https://docs.python.org/3/library/os.path.html#os.path.join)
    — build file paths portably.
-   [`pydantic.BaseModel`](https://docs.pydantic.dev/latest/concepts/models/)
    — declare validated data schemas.
-   [`FastAPI`](https://fastapi.tiangolo.com/) — high-performance web
    framework for Python APIs.
-   [`uvicorn`](https://www.uvicorn.org/) — ASGI server used to run FastAPI
    applications.
-   [`requests.post`](https://requests.readthedocs.io/en/latest/api/#requests.post)
    — send HTTP POST requests from Python.

➡️ Theory done. Time to build the class, then wrap it in a live API.

# Applied Exercises

## 2. Setup

Import the libraries and classes you'll use throughout. It is good practice
to keep all imports in a single cell at the top.

In this lesson you will iteratively edit `model.py` and `main.py` while
testing from this notebook. Enable **autoreload** so the notebook
automatically picks up your changes every time you save a file — no kernel
restart needed:

In [6]:
%load_ext autoreload
%autoreload 2

> 📌 `%autoreload 2` re-imports your edited modules before each cell runs.
> Without it, you'd have to restart the kernel after every change to
> `model.py` or `main.py` — autoreload is what makes the build-test-repeat
> loop in this lesson fast.

**Code 8.4.2.1**:

In [7]:
import os
import sqlite3
from glob import glob

import joblib
import pandas as pd
import requests
from arch.univariate.base import ARCHModelResult
from config import settings
from data import SQLRepository
from mock_alpha import activate_mock, deactivate_mock

## 3. Building the GarchModel class

### Problem

In previous lessons you wrote standalone functions for wrangling data,
fitting a GARCH model, and generating predictions. Those functions depend
on shared state — the ticker symbol, the database repository, and the
trained model object. Scattering this state across loose variables is
error-prone and hard to reuse. You need a single, self-contained class that
manages the full modeling lifecycle.

### Approach

You will build a `GarchModel` class inside the `model` module (`model.py`).
The class will be constructed incrementally: start with `__init__`, then add
methods one at a time.

The diagram below shows the complete class you are building. Each box
represents a method, and the arrows show the typical order in which they are
called:

    GarchModel
    ┌────────────────────────────────────────────┐
    │                                            │
    │  Attributes (set in __init__):             │
    │  ├── ticker : str         (e.g. "AMBU…")   │
    │  ├── repo : SQLRepository (DB connection)  │
    │  ├── use_new_data : bool  (fetch from API?)│
    │  ├── model_directory : str(from settings)  │
    │  ├── data : pd.Series     (None at start)  │
    │  └── model : ARCHModelResult (None)        │
    │                                            │
    │  Methods (build one at a time):            │
    │                                            │
    │  ┌──────────────────┐                      │
    │  │ 1. wrangle_data  │ n_observations: int  │
    │  │    Read DB ──►   │ → stores self.data   │
    │  │    compute       │   (pd.Series of      │
    │  │    returns       │    daily % returns)  │
    │  └────────┬─────────┘                      │
    │           ▼                                │
    │  ┌──────────────────┐                      │
    │  │ 2. fit           │ p: int, q: int       │
    │  │    Build GARCH   │ → stores self.model  │
    │  │    model & fit   │   (ARCHModelResult)  │
    │  └────────┬─────────┘                      │
    │           ▼                                │
    │  ┌──────────────────┐                      │
    │  │ 3. predict_      │ horizon: int         │
    │  │    volatility    │ → returns dict       │
    │  │    (calls        │   {"2025-03-21": 1.2,│
    │  │    __clean_      │    "2025-03-22": 1.3}│
    │  │    prediction)   │                      │
    │  └──────────────────┘                      │
    │                                            │
    │  ┌──────────────────┐                      │
    │  │ 4. dump          │ → saves self.model   │
    │  │    Save model    │   to .pkl file       │
    │  │    to disk       │   returns filepath   │
    │  └──────────────────┘                      │
    │                                            │
    │  ┌──────────────────┐                      │
    │  │ 5. load          │ → loads .pkl file    │
    │  │    Load model    │   into self.model    │
    │  │    from disk     │                      │
    │  └──────────────────┘                      │
    │                                            │
    └────────────────────────────────────────────┘

The typical workflow is:

-   **Training:** `wrangle_data` → `fit` → `dump`
-   **Prediction:** `load` → `predict_volatility`

Open `model.py` — you will see a module docstring describing the class. Your
job is to add the code below it, building one method at a time and testing
each from this notebook before moving on.

> 🧩 Notice this mirrors the `clean_prediction`, `wrangle_data`, and GARCH
> fitting you already wrote in Lesson 3 — you are not learning new
> *modelling*, you are **repackaging** proven code as methods on one object.

### Tasks

#### Step 1 — Create a SQLRepository for testing

Before building the class, create a `SQLRepository` connected to your SQLite
database. You will use this object to test every method you add to
`GarchModel`.

**Code Task 8.4.3.1**:

In [8]:
connection = sqlite3.connect(settings.db_name, check_same_thread=False)

# Instantiate the repository layer, injection-passing the database connection
repo = SQLRepository(connection)

print("repo type:", type(repo))

repo type: <class 'data.SQLRepository'>


In [9]:
%autoreload 0

#### Step 2 — Write the `__init__` method

Open `model.py` and create the `GarchModel` class with only the `__init__`
method for now. The constructor should:

1.  Accept three parameters: `ticker` (str), `repo` (SQLRepository), and
    `use_new_data` (bool).
2.  Store each as an instance attribute (`self.ticker`, `self.repo`,
    `self.use_new_data`).
3.  Set `self.model_directory` from `settings.model_directory`.
4.  Initialize `self.data` and `self.model` to `None` — later methods will
    populate these attributes. Setting them here avoids `AttributeError`s if
    someone accesses them before calling `wrangle_data` or `fit`.

Here is the skeleton to add inside `model.py`:

``` python
from __future__ import annotations

import pandas as pd
from config import settings
from data import SQLRepository


class GarchModel:

   def __init__(
        self,
        ticker: str,
        repo: SQLRepository,
        use_new_data: bool,
    ) -> None:
        self.ticker = ticker
        self.repo = repo
        self.use_new_data = use_new_data
        self.model_directory = MODEL_DIRECTORY
        self.data: pd.Series | None = None
        self.model = None
```

Replace the `...` with the correct values. Then import and instantiate the
class here to verify.

**Code Task 8.4.3.2**:

In [10]:

from model import GarchModel

gm_ambuja = GarchModel(
    ticker="AMBUJACEM.BSE",
    repo=repo,
    use_new_data=False,
)

print("type:", type(gm_ambuja))
print("ticker:", gm_ambuja.ticker)
print("use_new_data:", gm_ambuja.use_new_data)





type: <class 'model.GarchModel'>
ticker: AMBUJACEM.BSE
use_new_data: False


**Self-check:** You should see `type: <class 'model.GarchModel'>`,
`ticker: AMBUJACEM.BSE`, and `use_new_data: False`. If you get an
`ImportError`, make sure you saved `model.py`.

#### Checkpoint 1 — `__init__`

> 🧪 This Checkpoint confirms the constructor wired up every attribute: the
> ticker, the exact `repo` object, `use_new_data=False`, and the
> `model_directory` pulled from `settings`. Each later method leans on these.

In [11]:
assert gm_ambuja.ticker == "AMBUJACEM.BSE", (
    f"Expected 'AMBUJACEM.BSE', got '{gm_ambuja.ticker}'"
)
assert gm_ambuja.repo == repo, (
    "repo attribute does not match the SQLRepository"
)
assert not gm_ambuja.use_new_data, (
    "use_new_data should be False"
)
assert gm_ambuja.model_directory == (
    settings.model_directory
), (
    f"Expected '{settings.model_directory}', "
    f"got '{gm_ambuja.model_directory}'"
)
print("__init__ checks passed.")

__init__ checks passed.


> ✅ Passing means the skeleton of the class is sound — now flesh out its
> first real method.

#### Step 3 — Add the `wrangle_data` method

Now add a `wrangle_data` method to `GarchModel` in `model.py`. This method:

1.  Accepts `n_observations` (int).
2.  If `self.use_new_data` is `True`, fetches fresh data from the API and
    stores it in the database:
    1.  Create an `AlphaVantageAPI` instance.
    2.  Call its `.get_daily()` method with the ticker.
    3.  Insert the result into the database using `self.repo.insert_table()`
        — pass the ticker as `table_name`, the fetched data as `records`,
        and `if_exists="replace"`.
3.  Read the table from the database using `self.repo.read_table()` with the
    ticker as `table_name`.
4.  Sort the DataFrame by its index.
5.  Compute daily percentage returns from the `"close"` column using
    `.pct_change() * 100`.
6.  Drop NaN values, keep the last `n_observations` rows, and store the
    resulting `pd.Series` in `self.data`.

You will need these additional imports at the top of `model.py`:

``` python
import pandas as pd
from data import AlphaVantageAPI, SQLRepository
```

Test it below by wrangling 1000 observations. Because we pass
`use_new_data=True`, the method will try to call the AlphaVantage API. To
avoid hitting the real API (which requires a key and network access), we use
the `mock_alpha` helper from earlier lessons: `activate_mock()` replaces the
API call with a local copy of the data, and `deactivate_mock()` restores
normal behavior afterward.

**Code 8.4.3.3**:

In [12]:
model_test = GarchModel(
    ticker="AMBUJACEM.BSE",
    repo=repo,
    use_new_data=True,
)
activate_mock()
model_test.wrangle_data(n_observations=1000)
deactivate_mock()

print("data type:", type(model_test.data))
print("data shape:", model_test.data.shape)
model_test.data.head()

data type: <class 'pandas.Series'>
data shape: (1000,)


date
2022-02-25    0.651784
2022-02-28    1.780800
2022-03-02   -3.244791
2022-03-03   -4.619431
2022-03-04    0.827301
Name: return, dtype: float64

**Self-check:** `data type` should be
`<class 'pandas.core.series.Series'>` and `data shape` should be `(1000,)`.
The `.head()` output should show decimal numbers (daily percentage
returns).

> 📊 Note how `use_new_data` toggles the data *source* without changing the
> rest of the method: `True` re-fetches from the (mocked) API and refreshes
> the database; `False` reads straight from SQLite. The repository pattern
> from Lesson 2 is what makes that switch a one-line branch.

#### Checkpoint 2 — `wrangle_data`

> 🧪 Confirms `wrangle_data` populated `self.data` as a `pd.Series` of
> exactly the requested length (1000).

In [13]:
assert hasattr(model_test, "data"), (
    "After wrangle_data, model should have 'data'"
)
assert isinstance(model_test.data, pd.Series), (
    f"data should be pd.Series, "
    f"got {type(model_test.data)}"
)
assert model_test.data.shape == (1000,), (
    f"Expected shape (1000,), "
    f"got {model_test.data.shape}"
)
print("wrangle_data checks passed.")

wrangle_data checks passed.


> ✅ With data flowing into the class, give it the ability to model.

#### Step 4 — Add the `fit` method

Add a `fit` method that:

1.  Accepts `p` (int) and `q` (int) — the ARCH and GARCH orders.
2.  Builds a GARCH(p, q) model on `self.data` using the `arch_model`
    function from the `arch` library. Pass `rescale=False` so the library
    does not rescale your data (the returns are already in percentage form).
3.  Fits the model by calling `.fit()` with `disp="off"` to suppress
    optimizer output, and stores the fitted result in `self.model`.

You will need this additional import:

``` python
from arch import arch_model
```

Test by fitting a GARCH(1,1) model on the data you just wrangled.

**Code 8.4.3.4**:

In [14]:
model_test.fit(p=1, q=1)
print("model type:", type(model_test.model))
print("params:", model_test.model.params.index.tolist())
model_test.model.summary()

model type: <class 'arch.univariate.base.ARCHModelResult'>
params: ['mu', 'omega', 'alpha[1]', 'beta[1]']


<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - GARCH Model Results                      
==============================================================================
Dep. Variable:                 return   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -2043.66
Distribution:                  Normal   AIC:                           4095.31
Method:            Maximum Likelihood   BIC:                           4114.94
                                        No. Observations:                 1000
Date:                Sat, Jun 27 2026   Df Residuals:                      999
Time:                        21:02:29   Df Model:                            1
                                 Mean Model                                
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
mu         9.7082e-03  5.511e-02      0.176      0.860 [-9.831e-02,  0.118]
                              Volatility Model                             
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
omega          0.4793      0.253      1.898  5.775e-02 [-1.575e-02,  0.974]
alpha[1]       0.2174  6.858e-02      3.171  1.521e-03  [8.302e-02,  0.352]
beta[1]        0.6849  8.465e-02      8.092  5.890e-16    [  0.519,  0.851]
===========================================================================

Covariance estimator: robust
"""

**Self-check:** `model type` should be
`<class 'arch.univariate.base.ARCHModelResult'>` and `params` should show
`['mu', 'omega', 'alpha[1]', 'beta[1]']`.

> 🔍 Those four parameters are exactly the GARCH(1,1) terms from Lesson 3:
> `mu` (mean return), `omega` (baseline variance), `alpha[1]` (reaction to
> yesterday's shock), and `beta[1]` (carry-over of yesterday's variance).
> The class now holds a fully fitted model on `self.model`.

#### Checkpoint 3 — `fit`

> 🧪 Verifies `fit` stored an `ARCHModelResult` on `self.model`.

In [15]:
assert hasattr(model_test, "model"), (
    "After fit, model should have 'model' attribute"
)
assert isinstance(model_test.model, ARCHModelResult), (
    f"model should be ARCHModelResult, "
    f"got {type(model_test.model)}"
)
print("fit checks passed.")

fit checks passed.


> ✅ The model is trained. Next, turn it into a forecast a client can use.

#### Step 5 — Add the `predict_volatility` method

Add two methods: a private helper `__clean_prediction` that does the heavy
lifting, and a public `predict_volatility` that simply calls it.

> **Note on the `__` prefix:** naming a method with two leading underscores
> (e.g. `__clean_prediction`) is Python’s “name mangling” convention — it
> makes the method harder to call from outside the class, signaling that it
> is an internal implementation detail. You can also use a single
> underscore (`_clean_prediction`) if you prefer a simpler convention.

The `__clean_prediction` helper accepts `horizon` (int) and:

1.  Calls `self.model.forecast(horizon=horizon)` to get the raw forecast
    object.
2.  Extracts the variance values from the **last row** of
    `forecast.variance` (use `.iloc[-1]`).
3.  Gets the last date from `self.model.resid.index`.
4.  Generates a business-day date range of length `horizon` starting one day
    after that last date. Use `pd.bdate_range` with `pd.DateOffset(days=1)`
    to offset the start.
5.  Loops over the dates and variance values together. For each pair,
    converts the date to an ISO 8601 string (`.isoformat()`) and takes the
    square root of the variance (`np.sqrt`) to get volatility. Collects
    everything into a `dict` and returns it.

The `predict_volatility` method just accepts `horizon` (int) and delegates
to the helper.

You will need this additional import:

``` python
import numpy as np
```

Use this skeleton as a starting point — replace each `...` with the correct
code:

``` python
def __clean_prediction(
    self, horizon: int
) -> dict[str, float]:
    forecast = self.model.forecast(horizon=...)
    variance = forecast.variance.iloc[...]

    last_date = self.model.resid.index[...]
    dates = pd.bdate_range(
        start=last_date + pd.DateOffset(days=...),
        periods=...,
    )

    prediction = {}
    for i, date in enumerate(dates):
        prediction[date.isoformat()] = float(
            np.sqrt(variance.iloc[...])
        )
    return prediction

def predict_volatility(
    self, horizon: int
) -> dict[str, float]:
    return self.__clean_prediction(horizon=...)
```

Test with a 5-day horizon.

**Code Task 8.4.3.5**:

In [16]:
prediction = model_test.predict_volatility(
    horizon= 5
)
print("type:", type(prediction))
print("prediction:", prediction)

type: <class 'dict'>
prediction: {'2026-03-23T00:00:00': 2.525939884973802, '2026-03-24T00:00:00': 2.4973143115219445, '2026-03-25T00:00:00': 2.471199466997047, '2026-03-26T00:00:00': 2.447395602750832, '2026-03-27T00:00:00': 2.425715726770051}


**Self-check:** You should get a `dict` with 5 entries. Each key is an ISO
8601 date string (e.g., `"2025-03-21T00:00:00"`) and each value is a
positive float.

> 🧩 This is the Lesson 3 `clean_prediction` function, now a private method.
> Splitting the public `predict_volatility` from the private
> `__clean_prediction` keeps the *interface* (what callers use) separate
> from the *implementation* (how it's done) — a clean class can change the
> latter without breaking the former.

#### Checkpoint 4 — `predict_volatility`

> 🧪 Confirms the forecast is a `dict` of 5 string→float entries — the
> JSON-ready shape the API will serve.

In [17]:
assert isinstance(prediction, dict), (
    f"prediction should be dict, "
    f"got {type(prediction)}"
)
assert len(prediction) == 5, (
    f"Expected 5 entries, got {len(prediction)}"
)
assert all(
    isinstance(k, str) for k in prediction.keys()
), "All keys should be strings"
assert all(
    isinstance(v, float) for v in prediction.values()
), "All values should be floats"
print("predict_volatility checks passed.")

predict_volatility checks passed.


> ✅ The model can now forecast. The last two methods make it *persistent*.

#### Step 6 — Add the `dump` method

Add a `dump` method that:

1.  Creates `self.model_directory` if it does not exist (use `os.makedirs`
    with `exist_ok=True`).
2.  Gets the current timestamp as an ISO 8601 string (hint:
    `datetime.now().isoformat()`).
3.  Constructs the full file path by joining `self.model_directory` with a
    filename built from the timestamp and ticker
    (`{timestamp}_{ticker}.pkl`). Use `os.path.join` to build the path.
4.  Saves `self.model` to that path using `joblib.dump`.
5.  Returns the full file path as a string.

You will need these additional imports:

``` python
import os
from datetime import datetime
import joblib
```

**Code 8.4.3.6**:

In [18]:
filename = model_test.dump()
print("filename:", filename)
print("file exists:", os.path.exists(filename))

filename: models/2026-06-27T21:02:29.383488_AMBUJACEM.BSE.pkl
file exists: True


**Self-check:** The filename should look like
`models/2025-03-20T14:30:00.123456_AMBUJACEM.BSE.pkl` and the file should
exist on disk.

> 📌 The timestamp-prefixed filename is what makes `load` (next step) able to
> pick the newest model by simple sorting — and it keeps a full history of
> every training run rather than overwriting.

#### Checkpoint 5 — `dump`

> 🧪 Verifies `dump` returned a path string and the `.pkl` file actually
> exists on disk.

In [19]:
assert isinstance(filename, str), (
    f"dump should return str, got {type(filename)}"
)
assert os.path.exists(filename), (
    f"File not found at {filename}"
)
print("dump checks passed.")

dump checks passed.


> ✅ The model is saved. Now prove you can get it back.

#### Step 7 — Add the `load` method

Finally, add a `load` method that:

1.  Builds a glob pattern that matches all `.pkl` files for `self.ticker`
    inside `self.model_directory`. Use `os.path.join` to combine the
    directory with a pattern like `*{ticker}.pkl`.
2.  Passes that pattern to `glob()` and sorts the result. Sorting
    alphabetically works because filenames start with a timestamp, so the
    last entry is the most recent.
3.  If no files are found, raises an `Exception` with a helpful message.
4.  Loads the most recent file (the last one in the sorted list) with
    `joblib.load` and assigns the result to `self.model`.

You will need this additional import:

``` python
from glob import glob
```

Test by creating a **fresh** `GarchModel` instance (no training) and loading
the model you just saved.

**Code 8.4.3.7**:

In [20]:
model_test2 = GarchModel(
    ticker="AMBUJACEM.BSE",
    repo=repo,
    use_new_data=False,
)

# Before loading, model attribute should be None
print("model before load:", model_test2.model)

model_test2.load()

print("model type after load:", type(model_test2.model))
model_test2.model.summary()

model before load: None
model type after load: <class 'arch.univariate.base.ARCHModelResult'>


<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - GARCH Model Results                      
==============================================================================
Dep. Variable:                 return   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -2043.66
Distribution:                  Normal   AIC:                           4095.31
Method:            Maximum Likelihood   BIC:                           4114.94
                                        No. Observations:                 1000
Date:                Sat, Jun 27 2026   Df Residuals:                      999
Time:                        21:02:29   Df Model:                            1
                                 Mean Model                                
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
mu         9.7082e-03  5.511e-02      0.176      0.860 [-9.831e-02,  0.118]
                              Volatility Model                             
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
omega          0.4793      0.253      1.898  5.775e-02 [-1.575e-02,  0.974]
alpha[1]       0.2174  6.858e-02      3.171  1.521e-03  [8.302e-02,  0.352]
beta[1]        0.6849  8.465e-02      8.092  5.890e-16    [  0.519,  0.851]
===========================================================================

Covariance estimator: robust
"""

**Self-check:** Before `load()`, the model attribute should be `None`.
After `load()`, `model type` should be `ARCHModelResult` and `summary()`
should show the same parameters as before.

> 📊 This `None → ARCHModelResult` transition is the **dump/load round-trip
> closing**: a brand-new instance that never trained can still predict,
> because `load` restores the persisted model. That is precisely how the
> `/predict` endpoint will work — fresh `GarchModel`, `load`, forecast.

#### Checkpoint 6 — `load`

> 🧪 Confirms `load` restored an `ARCHModelResult` onto the fresh instance.

In [21]:
assert isinstance(
    model_test2.model, ARCHModelResult
), (
    f"After load, model should be ARCHModelResult, "
    f"got {type(model_test2.model)}"
)
print("load checks passed.")

load checks passed.


> ✅ All six methods work in isolation. The Final Checkpoint exercises them
> together.

### Final Checkpoint — GarchModel class

> 🧪 One gate over the whole class: constructor attributes, wrangled data
> shape, fitted model type, a 5-entry prediction dict, a saved file on disk,
> and a successfully reloaded model. Passing it certifies the entire
> `GarchModel` lifecycle end to end.

In [22]:
# --- Final Checkpoint: GarchModel class ---
assert gm_ambuja.ticker == "AMBUJACEM.BSE"
assert gm_ambuja.repo == repo
assert not gm_ambuja.use_new_data
assert (
    gm_ambuja.model_directory
    == settings.model_directory
)
assert isinstance(model_test.data, pd.Series)
assert model_test.data.shape == (1000,)
assert isinstance(
    model_test.model, ARCHModelResult
)
assert isinstance(prediction, dict)
assert len(prediction) == 5
assert isinstance(filename, str)
assert os.path.exists(filename)
assert isinstance(
    model_test2.model, ARCHModelResult
)

print("All GarchModel checks passed.")

All GarchModel checks passed.


> ✅ The class is production-ready. Everything left is about *exposing* it
> over HTTP.

➡️ A class only callable from Python isn't deployed. Wrap it in a web API.

## 4. Building the FastAPI application

### Problem

You have a fully functional `GarchModel` class, but it can only be used from
within a Python session. Other developers or services need a way to train
models and request forecasts programmatically — without importing your code
or having direct access to the database. You need a REST API that accepts
JSON requests and returns JSON responses.

### Why we need a `main.py` file

In previous lessons you wrote classes like `SQLRepository` and `GarchModel`
in separate `.py` modules (`data.py`, `model.py`) so they could be **reused
across notebooks**. The `main.py` file serves a different purpose: it is the
**entry point** for a web server. When you run `uvicorn main:app`, Python
loads `main.py` and starts serving HTTP requests. This file cannot live
inside a notebook — it must be a standalone `.py` file that `uvicorn` can
import.

Here is what `main.py` will contain and how each piece relates to what you
have already built:

    main.py
    ├── Imports
    │   ├── sqlite3, FastAPI, BaseModel  (standard / framework)
    │   ├── config.settings              (your config module)
    │   ├── data.SQLRepository           (your data module)
    │   └── model.GarchModel             (your model module)
    │
    ├── Pydantic data classes (request / response schemas)
    │   ├── FitIn        → what the client sends to /fit
    │   ├── FitOut       → what the server sends back from /fit
    │   ├── PredictIn    → what the client sends to /predict
    │   └── PredictOut   → what the server sends back from /predict
    │
    ├── build_model()    → helper: creates DB connection + GarchModel
    │
    └── FastAPI endpoints
        ├── GET  /hello   → health check
        ├── POST /fit     → train and save a model
        └── POST /predict → load a model and forecast

You will build this file **one piece at a time**, testing each piece from
the notebook before moving on.

### Approach

The overall request flow looks like this:

      ┌───────────┐   JSON request   ┌─────────────────────────┐
      │  Client   │ ──────────────► │  main.py                │
      │ (notebook │                 │                         │
      │  or curl) │                 │  /fit ──────────────────┤
      │           │                 │    1. build_model()     │
      │           │                 │    2. wrangle_data()    │
      │           │                 │    3. fit()             │
      │           │ ◄────────────── │    4. dump()            │
      │           │  JSON response  │    5. return FitOut     │
      │           │                 │                         │
      │           │                 │  /predict ──────────────┤
      │           │                 │    1. build_model()     │
      │           │                 │    2. load()            │
      │           │                 │    3. predict()         │
      │           │ ◄────────────── │    4. return PredictOut │
      └───────────┘  JSON response  └─────────────────────────┘

Each endpoint wraps its logic in a `try`/`except` block so that errors
produce a structured JSON response instead of crashing the server.

### Tasks

#### Step 1 — Create `main.py` with the initial imports

Open a new file called `main.py` in the same directory as your notebook.
Start with the following imports (you will add more as you go):

``` python
import sqlite3

from fastapi import FastAPI
from pydantic import BaseModel

from config import settings
from data import SQLRepository
from model import GarchModel
```

Save the file. You can verify that the imports work by running the cell
below. If you see a `ModuleNotFoundError`, make sure `main.py` is in the
same directory as `config.py`, `data.py`, and `model.py`.

**Code 8.4.4.1**:

In [23]:
import main
print("main.py imported successfully")

main.py imported successfully


#### Step 2 — Define the `FitIn` and `FitOut` data classes

When a client wants to train a model, it sends a JSON body with five fields.
You need a Pydantic class that validates those fields automatically. Add the
following two classes to `main.py`, **after** the imports:

| Class | Field | Type | Purpose |
|----------|-------------------|---------|-----------------------------------|
| `FitIn` | `ticker` | `str` | Stock ticker symbol |
|  | `use_new_data` | `bool` | Fetch fresh data from the API? |
|  | `n_observations` | `int` | How many return values to use |
|  | `p` | `int` | ARCH order |
|  | `q` | `int` | GARCH order |
| `FitOut` | *(inherits all `FitIn` fields)* |  |  |
|  | `success` | `bool` | Did training succeed? |
|  | `message` | `str` | Confirmation or error message |

`FitOut` should **inherit from `FitIn`** (just like `DemoOut` inherited from
`DemoIn` in the conceptual section). This guarantees the response always
echoes back the original request parameters.

Here is the skeleton — fill in the field definitions:

``` python
class FitIn(BaseModel):
    ticker: ...
    use_new_data: ...
    n_observations: ...
    p: ...
    q: ...


class FitOut(FitIn):
    success: ...
    message: ...
```

After saving `main.py`, run the cell below to verify. The autoreload
extension you enabled in Setup will pick up your changes automatically.

**Code Task 8.4.4.2**:

In [24]:
from main import FitIn, FitOut

# Instantiate the FitIn class with the specified validation constraints
fi = FitIn(
    ticker="AMBUJACEM.BSE",
    use_new_data=False,
    n_observations=2000,
    p=1,
    q=1,
)

print("FitIn:", fi)

FitIn: ticker='AMBUJACEM.BSE' use_new_data=False n_observations=2000 p=1 q=1


**Self-check:** Both print statements should display all fields with their
values. `FitOut` should show the five `FitIn` fields **plus** `success` and
`message`. If you get a `ValidationError`, check that the field names and
types match the table above exactly.

#### Checkpoint 1 — FitIn / FitOut

> 🧪 Confirms `FitIn` carries its five request fields, `FitOut` adds
> `success`/`message`, and crucially that `FitOut` **inherits** from `FitIn`
> (`isinstance(fo, FitIn)`) — the response-echoes-request contract.

In [25]:
assert hasattr(fi, "ticker"), (
    "FitIn missing 'ticker' field"
)
assert hasattr(fi, "use_new_data"), (
    "FitIn missing 'use_new_data' field"
)
assert hasattr(fi, "n_observations"), (
    "FitIn missing 'n_observations' field"
)
assert hasattr(fi, "p"), "FitIn missing 'p' field"
assert hasattr(fi, "q"), "FitIn missing 'q' field"
assert hasattr(fo, "success"), (
    "FitOut missing 'success' field"
)
assert hasattr(fo, "message"), (
    "FitOut missing 'message' field"
)
assert isinstance(fo, FitIn), (
    "FitOut should inherit from FitIn"
)
print("FitIn / FitOut checks passed.")

NameError: name 'fo' is not defined

> ✅ The training schema is defined. Repeat the pattern for prediction.

#### Step 3 — Define the `PredictIn` and `PredictOut` data classes

Now add two more data classes to `main.py` for the `/predict` endpoint:

| Class | Field | Type | Purpose |
|----------------|--------------|----------|-----------------------------------|
| `PredictIn` | `ticker` | `str` | Stock ticker symbol |
|  | `n_days` | `int` | Forecast horizon (business days) |
| `PredictOut` | *(inherits all `PredictIn` fields)* |  |  |
|  | `success` | `bool` | Did prediction succeed? |
|  | `forecast` | `dict` | Date strings to volatility |
|  | `message` | `str` | Confirmation or error message |

The pattern is the same: `PredictOut` inherits from `PredictIn`.

**Code Task 8.4.4.3**:

In [ ]:
from main import PredictIn, PredictOut

# Instantiate the PredictIn validation schema class for model forecasting
pi = PredictIn(ticker="AMBUJACEM.BSE", n_days=5)

print("PredictIn:", pi)

**Self-check:** `PredictIn` should show `ticker` and `n_days`. `PredictOut`
should show those two fields **plus** `success`, `forecast`, and `message`.

#### Checkpoint 2 — PredictIn / PredictOut

> 🧪 Same contract for prediction: `PredictIn` has the two request fields,
> `PredictOut` adds `success`/`forecast`/`message` and inherits from
> `PredictIn`.

In [ ]:
assert hasattr(pi, "ticker"), (
    "PredictIn missing 'ticker' field"
)
assert hasattr(pi, "n_days"), (
    "PredictIn missing 'n_days' field"
)
assert hasattr(po, "success"), (
    "PredictOut missing 'success' field"
)
assert hasattr(po, "forecast"), (
    "PredictOut missing 'forecast' field"
)
assert hasattr(po, "message"), (
    "PredictOut missing 'message' field"
)
assert isinstance(po, PredictIn), (
    "PredictOut should inherit from PredictIn"
)
print("PredictIn / PredictOut checks passed.")

> ✅ All four schemas exist. Next, a helper so both endpoints share one setup.

#### Step 4 — Write the `build_model` helper function

Both API endpoints need a `GarchModel` instance connected to the database.
Instead of repeating the same setup code in every endpoint, write a helper
function that does it once.

Add a function called `build_model` to `main.py` (after the data classes).
This is the same pattern you used in Section 3 (Steps 1–2) when you created a
`SQLRepository` and passed it to `GarchModel` — now wrapped in a reusable
function.

The function should:

1.  Accept `ticker` (str) and `use_new_data` (bool).
2.  Open a `sqlite3` connection to `settings.db_name` (remember to pass
    `check_same_thread=False`).
3.  Create a `SQLRepository` with that connection.
4.  Return a `GarchModel` instance configured with the ticker, repo, and
    `use_new_data`.
5.  Use a return type annotation of `-> GarchModel`.

**Code 8.4.4.4**:

In [ ]:
from main import build_model

model_shop = build_model(
    ticker="AMBUJACEM.BSE", use_new_data=False
)
print("type:", type(model_shop))
print("ticker:", model_shop.ticker)
print("repo type:", type(model_shop.repo))
print(
    "connection type:",
    type(model_shop.repo.connection),
)

**Self-check:** The type should be `<class 'model.GarchModel'>`, the ticker
should be `"AMBUJACEM.BSE"`, and the repo should be a `SQLRepository` with a
`sqlite3.Connection`.

> 🧱 `build_model` is the **DRY** (don't-repeat-yourself) keystone of the
> API: every endpoint gets its database-connected `GarchModel` from this one
> function, so a change to how models are constructed touches a single place.

#### Checkpoint 3 — build_model

> 🧪 Verifies `build_model` returns a `GarchModel` whose `repo` is a
> `SQLRepository` over a live `sqlite3.Connection`, configured for the right
> ticker.

In [ ]:
assert isinstance(model_shop.repo, SQLRepository), (
    "Expected SQLRepository, got "
    f"{type(model_shop.repo)}"
)
assert isinstance(
    model_shop.repo.connection, sqlite3.Connection
), (
    "Expected sqlite3.Connection on repo, got "
    f"{type(model_shop.repo.connection)}"
)
assert model_shop.ticker == "AMBUJACEM.BSE", (
    f"Expected 'AMBUJACEM.BSE', "
    f"got '{model_shop.ticker}'"
)
print("build_model checks passed.")

> ✅ The plumbing is ready. Now stand up the actual web server.

#### Step 5 — Create the FastAPI app and the `/hello` endpoint

Now you will create the actual web application. Add these lines to `main.py`
(after the `build_model` function):

``` python
app = FastAPI()


@app.get("/hello")
def hello():
    return {
        "message": "Hello from the volatility API!"
    }
```

The `app` object is what `uvicorn` will look for when you start the server.
The `/hello` endpoint is a simple health check — if it works, you know the
server is running.

**Starting the server:** Open a **separate terminal**, navigate to the
directory containing `main.py`, and run:

``` bash
uvicorn main:app --reload --workers 1 \
    --host localhost --port 8008
```

-   `main:app` tells uvicorn: “import the `app` object from `main.py`”.
-   `--reload` restarts the server automatically when you edit `main.py`.
-   `--workers 1` uses a single process (enough for development).

You should see output like `Uvicorn running on http://localhost:8008`.
**Keep this terminal open** — it is your running server.

**Tip — common issues when starting the server:**

-   `ModuleNotFoundError`: make sure the terminal’s working directory
    contains `main.py`, `model.py`, `data.py`, and `config.py`.
-   `Address already in use`: another process is using port 8008. Either
    stop it or choose a different port.

Now test the `/hello` endpoint from this notebook:

**Code 8.4.4.5**:

In [ ]:
url = "http://localhost:8008/hello"
response = requests.get(url=url)
print("status:", response.status_code)
print("body:", response.json())

**Self-check:** You should see `status: 200` and
`body: {'message': 'Hello from the volatility API!'}`. If you get a
`ConnectionError`, make sure the server is running in a separate terminal.

> 🚦 `/hello` is a **health check** — the simplest possible endpoint, with no
> modelling behind it. Confirming a `200` here isolates "is the server up
> and reachable?" from "is my modelling code correct?", so when the heavier
> endpoints misbehave you already know the transport works.

In [ ]:
# --- Checkpoint: /hello endpoint ---
assert response.status_code == 200, (
    "Expected status 200, "
    f"got {response.status_code}"
)
body = response.json()
assert "message" in body, (
    "Response missing 'message' key"
)
print("/hello endpoint check passed.")

> ✅ A passing health check means the server is live. Time for the real work.

#### Step 6 — Create the `/fit` endpoint

This is the most complex endpoint. It receives a `FitIn` request, trains a
GARCH model, saves it to disk, and returns a `FitOut` response. Add it to
`main.py` after the `/hello` endpoint.

Before you write the code, study the structure. Every POST endpoint in this
API follows the same pattern:

    @app.post("/path", response_model=OutputClass)
    def endpoint_name(request: InputClass) -> OutputClass:
        response = request.model_dump()   # ①
        try:
            ...                     # ② happy path
            response["success"] = True
            response["message"] = "..."
        except Exception as e:      # ③ error path
            response["success"] = False
            response["message"] = str(e)
        return OutputClass(**response)  # ④

Here is what each numbered part does:

1.  `request.model_dump()` converts the incoming Pydantic object into a
    plain dictionary. This becomes the starting point for the response (so
    the response echoes back all the original request fields).
2.  The `try` block contains the **happy path** — the actual work. If
    everything succeeds, you set `success` and `message` in the response
    dictionary.
3.  If anything goes wrong, the `except` block catches the error, sets
    `success=False`, and stores the error message as a string. This
    prevents the server from crashing.
4.  Finally, `OutputClass(**response)` unpacks the dictionary into a
    validated Pydantic object and returns it.

Now apply this pattern to the `/fit` endpoint. Use the skeleton below —
replace each `...` with the correct code:

``` python
@app.post("/fit", response_model=FitOut)
def fit_model(request: FitIn) -> FitOut:
    response = request.model_dump()
    try:
        model = build_model(
            ticker=...,
            use_new_data=...,
        )
        model.wrangle_data(n_observations=...)
        model.fit(p=..., q=...)
        filename = model.dump()
        response["success"] = ...
        response["message"] = ...
    except Exception as e:
        response["success"] = ...
        response["message"] = ...
    return FitOut(**response)
```

**Hints:**

-   All the values you need come from the `request` object
    (e.g. `request.ticker`, `request.p`).
-   Inside `try`, the four method calls are the exact same sequence you used
    in Section 3: `build_model` → `wrangle_data` → `fit` → `dump`.
-   For the success message, include the `filename` so the user knows where
    the model was saved.

Since `--reload` is active, the server picks up your changes automatically.
Send a POST request to train a GARCH(1,1) model:

**Code 8.4.4.6**:

In [ ]:
url = "http://localhost:8008/fit"
json_payload = {
    "ticker": "AMBUJACEM.BSE",
    "use_new_data": False,
    "n_observations": 2000,
    "p": 1,
    "q": 1,
}
response = requests.post(url=url, json=json_payload)
print("status:", response.status_code)
response.json()

**Self-check:** You should see `status: 200` and a JSON body with
`"success": true` and a message like
`"Trained and saved: models/2025-...AMBUJACEM.BSE.pkl"`. If `success` is
`false`, read the `message` field for the error — common causes are a
missing `stocks.sqlite` file or a typo in the ticker name.

> 📊 This single POST drove the entire **training pipeline over HTTP**:
> `build_model → wrangle_data → fit → dump`, the exact sequence you tested
> cell-by-cell in Section 3, now triggered by one JSON request. The
> `try`/`except` is why a bad ticker returns `success: false` with a message
> instead of a `500` crash.

In [ ]:
# --- Checkpoint: /fit endpoint ---
assert response.status_code == 200, (
    "Expected status 200, "
    f"got {response.status_code}"
)
fit_result = response.json()
assert fit_result["success"] is True, (
    "Expected success=True, "
    f"got {fit_result['success']}. "
    f"Server message: {fit_result.get('message')}"
)
assert fit_result["ticker"] == "AMBUJACEM.BSE", (
    "Expected ticker 'AMBUJACEM.BSE', "
    f"got '{fit_result['ticker']}'"
)
assert "message" in fit_result, (
    "Response missing 'message' field"
)
print("/fit endpoint check passed.")

> ✅ Training works end to end over the network. One endpoint left: serving
> the forecast.

**Tip:** If you get a Pydantic validation error (status 422), compare the
JSON keys you send against the field names in your `FitIn` class. They must
match exactly.

#### Step 7 — Create the `/predict` endpoint

The `/predict` endpoint loads a previously saved model and generates a
volatility forecast. Now it is your turn to write an endpoint **from
scratch**, following the same pattern you used for `/fit`.

Use these guidelines:

-   **Decorator:** `@app.post("/predict", response_model=PredictOut)`.
-   **Function signature:** accept a `PredictIn` request and return a
    `PredictOut`.
-   **Same structure as `/fit`:** start with `request.model_dump()`, wrap
    the work in `try`/`except`, and return `PredictOut(**response)` at the
    end.

Inside the `try` block, the happy path has three steps (compared to four in
`/fit`):

1.  Build a `GarchModel` using `build_model`. Prediction never fetches new
    data, so always pass `use_new_data=False`.
2.  Call `.load()` to restore the most recently saved model for that ticker
    from disk.
3.  Call `.predict_volatility()` using `request.n_days` as the horizon.

On success, store the forecast dictionary in `response["forecast"]`
alongside `success` and `message`. On failure, set `response["forecast"]` to
an empty dict `{}` so the response still has the correct shape.

**Important:** This endpoint will fail if you have not previously trained and
saved a model via `/fit`. The model file must exist on disk.

> **Hint:** Compare `PredictOut`’s fields (`success`, `forecast`,
> `message`) with `FitOut`’s (`success`, `message`). The only difference in
> the `except` block is that you also need to set `forecast` to `{}`.

Test it now by requesting a 5-day volatility forecast:

**Code 8.4.4.7**:

In [ ]:
url = "http://localhost:8008/predict"
json_payload = {
    "ticker": "AMBUJACEM.BSE",
    "n_days": 5,
}
response = requests.post(url=url, json=json_payload)
print("status:", response.status_code)
response.json()

**Self-check:** You should see `status: 200` with `"success": true` and a
`forecast` dictionary containing 5 entries. Each key is an ISO 8601 date
string and each value is a float representing predicted volatility.

> 📊 The `/predict` path is the mirror image of `/fit`:
> `build_model → load → predict_volatility`. It never retrains — it reloads
> the model `/fit` persisted and returns volatilities in the same
> date→float shape your Lesson 3 `clean_prediction` produced. **This is the
> payoff of the whole project**: a trained GARCH model, served as JSON, over
> HTTP, to any client.

In [ ]:
# --- Checkpoint: /predict endpoint ---
assert response.status_code == 200, (
    "Expected status 200, "
    f"got {response.status_code}"
)
predict_result = response.json()
assert predict_result["success"] is True, (
    "Expected success=True, "
    f"got {predict_result['success']}. "
    f"Server message: {predict_result.get('message')}"
)
assert isinstance(
    predict_result["forecast"], dict
), (
    "Expected forecast to be a dict, "
    f"got {type(predict_result['forecast'])}"
)
assert len(predict_result["forecast"]) == 5, (
    "Expected 5 forecast entries, "
    f"got {len(predict_result['forecast'])}"
)
assert all(
    isinstance(v, float)
    for v in predict_result["forecast"].values()
), "All forecast values should be floats"
print("/predict endpoint check passed.")

> ✅ Both endpoints pass. The deployment is complete and working end to end.

#### Step 8 — Review your complete `main.py`

At this point your `main.py` should contain all of the following components.
Use this checklist to verify you have everything:

-   [ ] Imports: `sqlite3`, `FastAPI`, `BaseModel`, `settings`,
    `SQLRepository`, `GarchModel`
-   [ ] `FitIn` class with 5 fields (`ticker`, `use_new_data`,
    `n_observations`, `p`, `q`)
-   [ ] `FitOut` class inheriting from `FitIn` with 2 extra fields
    (`success`, `message`)
-   [ ] `PredictIn` class with 2 fields (`ticker`, `n_days`)
-   [ ] `PredictOut` class inheriting from `PredictIn` with 3 extra fields
    (`success`, `forecast`, `message`)
-   [ ] `build_model` function returning a `GarchModel`
-   [ ] `app = FastAPI()`
-   [ ] `GET /hello` endpoint
-   [ ] `POST /fit` endpoint with try/except
-   [ ] `POST /predict` endpoint with try/except

### Final Checkpoint — FastAPI application

> 🧪 The capstone gate re-validates every API component at once: all four
> Pydantic schemas (fields + inheritance) and the `build_model` helper. Pass
> it and your `main.py` is complete and correct.

In [ ]:
# --- Checkpoint: FastAPI data classes and build_model ---
# FitIn / FitOut
assert hasattr(fi, "ticker"), (
    "FitIn missing 'ticker' field"
)
assert hasattr(fi, "use_new_data"), (
    "FitIn missing 'use_new_data' field"
)
assert hasattr(fi, "n_observations"), (
    "FitIn missing 'n_observations' field"
)
assert hasattr(fi, "p"), "FitIn missing 'p' field"
assert hasattr(fi, "q"), "FitIn missing 'q' field"
assert hasattr(fo, "success"), (
    "FitOut missing 'success' field"
)
assert hasattr(fo, "message"), (
    "FitOut missing 'message' field"
)
assert isinstance(fo, FitIn), (
    "FitOut should inherit from FitIn"
)

# PredictIn / PredictOut
assert hasattr(pi, "ticker"), (
    "PredictIn missing 'ticker' field"
)
assert hasattr(pi, "n_days"), (
    "PredictIn missing 'n_days' field"
)
assert hasattr(po, "success"), (
    "PredictOut missing 'success' field"
)
assert hasattr(po, "forecast"), (
    "PredictOut missing 'forecast' field"
)
assert hasattr(po, "message"), (
    "PredictOut missing 'message' field"
)
assert isinstance(po, PredictIn), (
    "PredictOut should inherit from PredictIn"
)

# build_model
assert isinstance(model_shop.repo, SQLRepository), (
    "Expected SQLRepository, got "
    f"{type(model_shop.repo)}"
)
assert isinstance(
    model_shop.repo.connection, sqlite3.Connection
), (
    "Expected sqlite3.Connection on repo, got "
    f"{type(model_shop.repo.connection)}"
)
assert model_shop.ticker == "AMBUJACEM.BSE", (
    f"Expected 'AMBUJACEM.BSE', "
    f"got '{model_shop.ticker}'"
)

print("All FastAPI component checks passed.")

# Wrap-up

In this lesson you completed the full deployment pipeline for the volatility
forecasting system:

-   Organised all modeling logic into a reusable `GarchModel` class with
    methods for data wrangling, fitting, predicting, saving, and loading.
-   Used `joblib` to serialize trained GARCH models to disk and load them
    back for prediction.
-   Defined Pydantic data classes (`FitIn`, `FitOut`, `PredictIn`,
    `PredictOut`) to validate API request and response schemas.
-   Built a FastAPI application with `/fit` and `/predict` endpoints that
    train models and serve forecasts over HTTP.
-   Sent HTTP requests from Python to interact with your running API and
    verified the results.

> 🧠 The idea to carry forward: **deployment is repackaging, not rewriting.**
> Every line of modelling logic here came from Lessons 1–3 — the new work
> was wrapping it in a class, persisting it with `joblib`, guarding it with
> Pydantic schemas, and exposing it through FastAPI. That journey from
> notebook experiment to network service is the same one professional teams
> take.

This concludes Project 8 and the Data Science Lab. You have now experienced
the complete lifecycle of a data science project: from loading and exploring
raw data, through statistical modeling and testing, to packaging your work
as a production-ready API. These are the same steps used in professional data
science teams to move from notebook experiments to deployed systems.